# Building Reliable Agents with Memory and Compaction

This Cookbook shows how to build an evidence review agent that can work through a long-running compliance investigation with the Agents SDK memory and compaction primitives.

We will start with a simple sandbox agent, then add two primitives:

- **Compaction** keeps long-running conversations manageable by carrying forward the state needed for later turns while reducing context size.
- **Memory** lets future sandbox-agent runs reuse durable workflow lessons from prior runs, without replaying every previous turn.

The important reliability idea is that these primitives solve different problems. Compaction helps the current run continue. Memory helps later runs start smarter. The generated memo remains the human-reviewed source of truth for the investigation.

References used for this notebook:

- [Agents SDK Guide](https://developers.openai.com/api/docs/guides/agents)
- [Sandbox Agents Guide](https://developers.openai.com/api/docs/guides/agents/sandboxes)
- [Compaction Guide](https://developers.openai.com/api/docs/guides/compaction)
- [Memory Guide](https://developers.openai.com/api/docs/guides/agents/sandboxes#persist-memory-across-runs)


## What You'll Build

By the end of the notebook, you will have an **Evidence Review Agent for a compliance investigation** with four stages:

1. A simple sandbox agent that reviews staged documents and writes a reviewer-ready artifact.
2. The same agent with `Compaction()` attached, so long reviews can continue after context grows.
3. The agent with SDK `Memory()` attached, so future runs can reuse process lessons.
4. A final configuration that uses both primitives and produces an evidence review memo.

This notebook keeps model execution opt-in. The code cells define runnable configurations, but `RUN_AGENT` defaults to `False` so readers can inspect the architecture without making API calls.


## Prerequisites

To run the live agent workflow, you need:

- Python 3.10 or later.
- The `openai-agents` package.
- An OpenAI API key available as `OPENAI_API_KEY`.
- A local Unix-like environment for `UnixLocalSandboxClient`. The notebook uses synthetic files created from the sandbox `Manifest`, so no external dataset is required.

The notebook is safe to inspect without credentials because `RUN_AGENT` defaults to `False`. Set `RUN_AGENT = True` only when you want to execute the model-backed sandbox run.


## Use Case: Evidence Review Agent for a Compliance Investigation

The scenario is a compliance team reviewing a set of enterprise documents in batches. The team wants the agent to inspect the documents, preserve source citations in its memo, track uncertainty in plain language, and generate a long-running artifact such as a status memo for human review.

This use case is a strong fit for memory and compaction because it has the same shape as many production investigative workflows:

- **Multiple document batches:** new evidence arrives over time, so early conclusions may be upgraded, contradicted, or superseded.
- **A sandbox workspace:** the agent needs files, a manifest, notes, and generated artifacts, not only prompt context.
- **Long-running review:** each batch may require many model turns, file reads, and tool calls.
- **Artifact generation:** the final output is a reviewer-ready memo, timeline, or escalation packet.
- **Human review:** the agent should preserve citations, uncertainty, and follow-up questions so a person can audit the result.

In this notebook, the documents are small synthetic compliance notes. In a production version, the same pattern can point at a KYC case folder, a support export, an incident archive, a diligence data room, or another document corpus with staged evidence.


## Folder Structure and Manifest

Before adding compaction or memory, decide what the agent's workspace should look like. Reliable long-running agents work better when the task state is visible as files instead of hidden in prompt text.

For the Evidence Review Agent, use a workspace structure like this:

```text
/workspace/
  README.md                         # Task instructions and evidence rules
  manifest.csv                      # Batch, doc_id, path, and description for each document
  docs/
    batch_1/                        # Initial evidence
    batch_2/                        # Follow-up or contradicting evidence
    batch_3/                        # Remediation or final evidence
  outputs/
    compliance_review_memo.md       # Reviewer-ready artifact
  memories/                         # Generated by SDK Memory()
  sessions/                         # Generated by SDK Memory()
```

The key design choice is that **the folder structure carries the work contract**. The prompt tells the agent how to behave, but the workspace tells it where inputs, generated artifacts, and generated memory files live.


### The Manifest Feature

In the Agents SDK, a `Manifest` is the fresh-session workspace contract for a sandbox agent. It describes the files, directories, mounts, environment, users, groups, and related workspace configuration that should exist when a new sandbox session starts.

The local SDK implementation defines these core fields:

| Manifest field | What it controls | How to use it in this Cookbook |
|---|---|---|
| `root` | Workspace root path. Defaults to `/workspace`. | Keep the default unless a sandbox provider expects a different root. |
| `entries` | Files, directories, local files, local directories, repos, or mounts to materialize. | Put `README.md`, `manifest.csv`, input documents, and `outputs/` here. |
| `environment` | Environment variables available when the sandbox starts. | Use only for non-secret runtime configuration. Keep credentials out of prompts and committed notebooks. |
| `users` / `groups` | Sandbox-local OS accounts and groups for providers that support them. | Usually unnecessary for a Cookbook, useful for production isolation. |
| `extra_path_grants` | Additional path grants, especially useful for Unix-local workflows. | Use sparingly when a sandbox needs scoped read/write access to host paths. |
| `remote_mount_command_allowlist` | Commands allowed against remote mounts. | Keep narrow when mounting external storage or data rooms. |

Manifest entry paths should be workspace-relative. Avoid absolute paths and `..` escapes so the same agent can move between Unix-local, Docker, and hosted sandbox providers.


### Folder and Manifest Best Practices

- Put source documents, manifests, helper files, and output directories in the `Manifest` instead of pasting large content into the prompt.
- Put longer task instructions in workspace files such as `README.md`, `task.md`, or `AGENTS.md`; keep agent instructions focused on behavior and boundaries.
- Use stable document IDs and a machine-readable manifest file so generated memos can cite sources and reviewers can inspect the path back to evidence.
- Let `Memory()` manage its own memory artifacts. By default, sandbox memory uses `memories/` and `sessions/` under the workspace.
- Keep generated artifacts under `outputs/` so the application can inspect, copy, validate, or archive them after the run.
- Keep mount scopes narrow. If you mount a data room, mount only what the agent should read or write.
- Treat secrets as runtime configuration injected by your application or sandbox provider, not as prompt text or committed manifest content.
- Prefer a small synthetic `File(...)` or `Dir(...)` entry for a tutorial, then switch to `LocalDir`, `GitRepo`, or storage mounts for production-sized datasets.


In [ ]:
# This tree is the design contract we will materialize with Manifest in the next cell.
WORKSPACE_TREE = """
/workspace/
  README.md
  manifest.csv
  docs/
    batch_1/
    batch_2/
    batch_3/
  outputs/
  memories/    # Generated by Memory()
  sessions/    # Generated by Memory()
""".strip()

print(WORKSPACE_TREE)


## Setup

Set `RUN_AGENT = True` only when you have installed the Agents SDK, configured your OpenAI API key, and are ready to make model calls. The expected output shape later in the notebook shows the intended artifact without requiring a live run.

This notebook defaults to `DISABLE_TRACING = True`. That avoids trace-ingestion errors for zero-data-retention organizations and is a conservative default for evidence-review workflows, where prompts, file reads, and tool outputs may contain sensitive material. If you are not in a ZDR organization, or you are only testing with this synthetic ACME data and want normal Agents SDK observability, set `DISABLE_TRACING = False`. In production, tracing or equivalent observability is usually useful, but it should be paired with an approved redaction, retention, access-control, and audit policy.


In [ ]:
from __future__ import annotations

import json
import os
import textwrap
from pathlib import Path

RUN_AGENT = False
MODEL = "gpt-5.5"
COMPACTION_MODEL = "gpt-5.4-mini"
WORKFLOW_NAME = "evidence-review-memory-compaction"
FORCE_COMPACTION_CHECKPOINT = True
DISABLE_TRACING = True

if DISABLE_TRACING:
    os.environ["OPENAI_AGENTS_DISABLE_TRACING"] = "true"
else:
    os.environ.pop("OPENAI_AGENTS_DISABLE_TRACING", None)

if RUN_AGENT and not os.environ.get("OPENAI_API_KEY"):
    raise RuntimeError("Set OPENAI_API_KEY before running the live sandbox workflow.")

print({
    "run_agent": RUN_AGENT,
    "model": MODEL,
    "compaction_model": COMPACTION_MODEL,
    "force_compaction_checkpoint": FORCE_COMPACTION_CHECKPOINT,
    "tracing_disabled": DISABLE_TRACING,
})


## Prepare a Small Evidence Workspace

A `Manifest` describes the starting files in a fresh sandbox workspace. For this tutorial, the workspace includes:

- a `manifest.csv` listing documents by batch and document ID,
- three small document batches,
- an output directory for the review memo.

The only memory primitive we attach later is the SDK's SDK `Memory()` capability. Investigation findings stay in the generated reviewer memo, where they can be cited and inspected.


In [ ]:
from agents.sandbox import Manifest
from agents.sandbox.entries import Dir, File


def workspace_file(text: str) -> File:
    return File(content=textwrap.dedent(text).strip().encode("utf-8") + b"\n")


def build_evidence_manifest() -> Manifest:
    return Manifest(
        entries={
            "README.md": workspace_file(
                """
                # Evidence Review Workspace

                Review the documents in batch order. Cite document IDs from
                `manifest.csv` when making findings. Write the final memo to
                `outputs/compliance_review_memo.md`.
                """
            ),
            "manifest.csv": workspace_file(
                """
                doc_id,batch,path,description
                ACME-B1-001,1,docs/batch_1/payment_policy.txt,Baseline payment policy
                ACME-B1-002,1,docs/batch_1/vendor_exception.txt,Vendor exception note
                ACME-B2-001,2,docs/batch_2/audit_followup.txt,Audit follow-up request
                ACME-B2-002,2,docs/batch_2/approval_thread.txt,Approval clarification
                ACME-B3-001,3,docs/batch_3/remediation_plan.txt,Remediation plan
                """
            ),
            "docs/batch_1/payment_policy.txt": workspace_file(
                """
                doc_id: ACME-B1-001
                ACME requires two approvals for payments over $50,000. Exceptions must
                be logged with Finance Ops and reviewed within five business days.
                """
            ),
            "docs/batch_1/vendor_exception.txt": workspace_file(
                """
                doc_id: ACME-B1-002
                A vendor onboarding exception was approved verbally for Northwind
                Logistics because the renewal was time-sensitive. The note does not show
                a Finance Ops log entry.
                """
            ),
            "docs/batch_2/audit_followup.txt": workspace_file(
                """
                doc_id: ACME-B2-001
                Internal Audit asked Finance Ops to confirm whether Northwind Logistics
                received post-approval review. The request says missing exception logs
                should be treated as a control gap until resolved.
                """
            ),
            "docs/batch_2/approval_thread.txt": workspace_file(
                """
                doc_id: ACME-B2-002
                The approval thread says Legal approved the vendor exception, but Finance
                Ops approval was still pending when the payment was released.
                """
            ),
            "docs/batch_3/remediation_plan.txt": workspace_file(
                """
                doc_id: ACME-B3-001
                The remediation plan requires Finance Ops to reconcile all verbal vendor
                exceptions from Q4 and add retrospective control attestations.
                """
            ),
            "outputs": Dir(),
        }
    )


manifest = build_evidence_manifest()
print(f"Workspace entries: {len(manifest.entries)}")


## Step 1: Start With a Simple Agent Configuration

First, build the agent without memory or compaction. The goal is to make the baseline behavior clear before adding primitives.

One subtle point: `SandboxAgent` defaults can include built-in capabilities. To keep this baseline explicit, pass the exact capability list you want. Here we include only the workspace tools the agent needs to inspect files and write an artifact: `Filesystem()` and `Shell()`. We intentionally do **not** attach `Compaction()` or `Memory()` yet.

- `Filesystem()` gives the sandbox agent file-oriented workspace access so it can read staged evidence and write the memo artifact. In the Sandbox Agents guide, capabilities are described as the way to attach sandbox-native behavior and tools to a `SandboxAgent`: https://developers.openai.com/api/docs/guides/agents/sandboxes#give-the-agent-capabilities
- `Shell()` lets the agent inspect the workspace with terminal commands such as listing files, opening evidence documents, and searching for terms across batches. The Sandbox Agents guide notes that `Shell()` is one of the default capabilities, and the Shell tool guide explains that shell gives models a terminal environment for hosted or local execution: https://developers.openai.com/api/docs/guides/agents/sandboxes#give-the-agent-capabilities and https://developers.openai.com/api/docs/guides/tools-shell
- For this baseline, these two capabilities are enough: `Filesystem()` handles workspace reads and writes, while `Shell()` handles deterministic inspection and search. Memory and compaction are added only after the baseline harness is clear.


In [ ]:
from agents.sandbox import SandboxAgent
from agents.sandbox.capabilities import Filesystem, Shell

BASELINE_INSTRUCTIONS = """
You are an evidence review agent for a compliance investigation.

Review documents in batch order. Keep these boundaries clear:
- Cite document IDs from `manifest.csv` for each finding.
- If evidence is incomplete, record an open question instead of guessing.
- Write a concise reviewer memo to `outputs/compliance_review_memo.md`.
- Use the generated memo as the reviewer-facing investigation artifact.
""".strip()


def build_baseline_agent() -> SandboxAgent:
    return SandboxAgent(
        name="Evidence Review Agent",
        model=MODEL,
        instructions=BASELINE_INSTRUCTIONS,
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
        ],
    )


baseline_agent = build_baseline_agent()
print([type(capability).__name__ for capability in baseline_agent.capabilities])


In [ ]:
from agents import Runner
from agents.run import RunConfig
from agents.sandbox import SandboxRunConfig
from agents.sandbox.sandboxes.unix_local import UnixLocalSandboxClient

BASELINE_TASK = """
Review Batch 1 only, then draft `outputs/compliance_review_memo.md` for a
compliance reviewer. Include cited findings and open questions.
""".strip()


async def _read_text_file(session, path: str) -> str | None:
    try:
        handle = await session.read(Path(path))
    except Exception as exc:
        if "NotFound" not in type(exc).__name__:
            raise
        return None
    try:
        return handle.read().decode("utf-8", errors="replace")
    finally:
        handle.close()


async def _list_workspace_files(session) -> str:
    result = await session.exec(
        "find outputs memories sessions -maxdepth 4 -type f 2>/dev/null | sort || true",
        timeout=30,
    )
    return result.stdout.decode("utf-8", errors="replace").strip()


async def _read_memory_artifacts(session) -> dict[str, str]:
    memory_artifacts = {}
    for path in ["memories/MEMORY.md", "memories/memory_summary.md"]:
        text = await _read_text_file(session, path)
        if text:
            memory_artifacts[path] = text
    return memory_artifacts


async def run_in_unix_sandbox(agent: SandboxAgent, task: str, *, sdk_session=None) -> dict[str, object]:
    client = UnixLocalSandboxClient()
    session = await client.create(manifest=agent.default_manifest)
    try:
        await session.start()
        result = await Runner.run(
            agent,
            task,
            max_turns=12,
            run_config=RunConfig(
                sandbox=SandboxRunConfig(session=session),
                workflow_name=WORKFLOW_NAME,
                tracing_disabled=DISABLE_TRACING,
            ),
            session=sdk_session,
        )

        compaction_checkpoint = None
        if sdk_session is not None and FORCE_COMPACTION_CHECKPOINT:
            compaction_checkpoint = await force_compaction_checkpoint(sdk_session)

        # Memory generation runs as a sandbox pre-stop hook. Flush it before reading artifacts
        # so `memories/MEMORY.md` and `memories/memory_summary.md` are available here.
        await session.run_pre_stop_hooks()

        memo = await _read_text_file(session, "outputs/compliance_review_memo.md")
        files = await _list_workspace_files(session)
        memory_artifacts = await _read_memory_artifacts(session)

        return {
            "result": result,
            "final_output": str(result.final_output),
            "memo": memo,
            "workspace_files": files,
            "memory_artifacts": memory_artifacts,
            "compaction_checkpoint": compaction_checkpoint,
        }
    finally:
        await session.aclose()


if RUN_AGENT:
    baseline_run = await run_in_unix_sandbox(baseline_agent, BASELINE_TASK)
    print(baseline_run["final_output"])
    print("\n----- END AGENT OUTPUT -----")
else:
    print("RUN_AGENT is False. Baseline agent is configured but not executed.")


## Step 2: Add Compaction

Compaction is for long-running work. As a conversation grows, compaction reduces context size while preserving the state needed for later turns. There are three useful ways to think about it:

1. **Automatic compaction with `Compaction()`**: attach the capability and let the SDK compact when context pressure requires it. This is the simplest production default.
2. **Threshold-based compaction with `StaticCompactionPolicy`**: set a lower or explicit threshold so compaction is easier to trigger in tests, demos, or constrained deployments.
3. **Forced checkpoint compaction with `OpenAIResponsesCompactionSession.run_compaction({"force": True})`**: compact at an application-defined phase boundary, such as after Batch 2 and before Batch 3.

This workflow uses option 3 because the evidence set is intentionally small and the investigation has a natural phase boundary. A forced checkpoint makes compaction explicit at the point where the application wants to preserve working context before continuing.

Best practices:

- Do not use compacted context as the source of truth for facts. Keep citations visible in generated artifacts such as the memo.
- Use automatic compaction for ordinary long-running agent runs unless you need a specific phase checkpoint.
- Use threshold policies for tests and controlled environments where you need predictable context limits.
- Use forced checkpoints when your workflow has meaningful boundaries, such as batches, stages, or review gates.


In [ ]:
from agents.sandbox.capabilities import Compaction, StaticCompactionPolicy


def build_compaction_agent(*, demo_threshold: int | None = None) -> SandboxAgent:
    if demo_threshold is None:
        compaction = Compaction()
    else:
        compaction = Compaction(policy=StaticCompactionPolicy(threshold=demo_threshold))

    return SandboxAgent(
        name="Evidence Review Agent with Compaction",
        model=MODEL,
        instructions=(
            BASELINE_INSTRUCTIONS
            + "\n\nWhen context is compacted, preserve the current batch, cited facts, open "
            "questions, artifact paths, and unresolved reviewer concerns."
        ),
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
            compaction,
        ],
    )


compaction_agent = build_compaction_agent()
threshold_compaction_agent = build_compaction_agent(demo_threshold=8_000)
print({
    "automatic": [type(capability).__name__ for capability in compaction_agent.capabilities],
    "threshold_policy": [type(capability).__name__ for capability in threshold_compaction_agent.capabilities],
})


### How Compaction Gets Triggered

With the `Compaction()` capability, server-side compaction is eligible to run when the active context grows large enough. That is the current default behavior: attach the capability and let the SDK manage context pressure.

For small tutorials, automatic compaction can be hard to see because the run may never get close to the model context limit. A lower `StaticCompactionPolicy` can help, but it still depends on the rendered context crossing the threshold.

For a small evidence set, a forced checkpoint is the clearest operational pattern. The `OpenAIResponsesCompactionSession` wrapper stores session history and lets the application call `run_compaction({"force": True})` at a phase boundary. That makes compaction visible without inflating the evidence set.


In [ ]:
from agents import OpenAIResponsesCompactionSession, SQLiteSession


def build_compaction_session() -> OpenAIResponsesCompactionSession:
    underlying = SQLiteSession("evidence_review_session.sqlite")
    return OpenAIResponsesCompactionSession(
        session_id="evidence-review-demo",
        underlying_session=underlying,
        model=COMPACTION_MODEL,
        compaction_mode="input",
    )


async def force_compaction_checkpoint(session: OpenAIResponsesCompactionSession) -> dict[str, int]:
    items_before = await session.get_items()
    await session.run_compaction({"force": True, "compaction_mode": "input"})
    items_after = await session.get_items()
    return {"items_before": len(items_before), "items_after": len(items_after)}


print("Compaction session helper defined. The final run uses it to show an explicit phase checkpoint.")


## Step 3: Attach Memory

Memory is for reuse across runs. Sandbox memory distills useful lessons from prior workspace runs into memory artifacts that later runs can read. It is separate from SDK session history and separate from compaction.

For this Cookbook, use the SDK memory primitive with a small generation prompt:

```python
Memory(generate=MemoryGenerateConfig(extra_prompt=...))
```

That prompt steers the memory generator. It does **not** change the compliance-review prompt itself. Instead, it tells the SDK memory-generation phase what kind of information should become reusable memory.

The `Memory()` capability writes SDK-managed memory artifacts into the sandbox workspace. In this example, the SDK produces files such as `memories/MEMORY.md` and `memories/memory_summary.md`. That file layout is standard SDK sandbox memory behavior; the notebook-specific part is the `MemoryGenerateConfig.extra_prompt`, which steers what kind of information should be stored.

For this use case, memory should preserve workflow lessons, not compliance findings. Good memory examples:

- "The reviewer prefers a one-page memo with cited findings and open questions."
- "Always separate verbal-approval evidence from Finance Ops approval evidence."
- "When evidence is incomplete, preserve the uncertainty in the memo instead of guessing."

Best practices:

- Start with `Memory()` for the default memory behavior, then add `MemoryGenerateConfig.extra_prompt` when the application needs a clearer write policy.
- Include `Shell()` because memory reads require shell search/open behavior.
- Include `Filesystem()` when live memory updates or repairs should be allowed.
- Let memory store reusable process knowledge. Keep document-specific findings in the generated memo and supporting artifacts.


In [ ]:
from agents.sandbox import MemoryGenerateConfig
from agents.sandbox.capabilities import Memory

MEMORY_GENERATION_PROMPT = """
Store reusable workflow lessons only.
Do not store ACME-specific compliance findings, document facts, evidence citations,
or memo conclusions. Those belong in outputs/compliance_review_memo.md.
Memory should help future evidence-review workflows behave better; it should not
become a second investigation record.
""".strip()


def workflow_memory() -> Memory:
    return Memory(
        generate=MemoryGenerateConfig(
            extra_prompt=MEMORY_GENERATION_PROMPT,
        )
    )


def build_memory_agent() -> SandboxAgent:
    return SandboxAgent(
        name="Evidence Review Agent with Memory",
        model=MODEL,
        instructions=BASELINE_INSTRUCTIONS,
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
            workflow_memory(),
        ],
    )


memory_agent = build_memory_agent()
print([type(capability).__name__ for capability in memory_agent.capabilities])


## Step 4: Run With Both Compaction and Memory

The final configuration combines the two primitives:

- `Compaction()` keeps the active long-running review manageable as the context grows.
- `Memory(generate=MemoryGenerateConfig(...))` captures reusable workflow lessons for future reviews while avoiding document-specific investigation facts.
- The memo in `outputs/` remains the reviewer-facing artifact.

This separation gives the agent enough continuity to work through multiple batches while keeping the implementation focused on built-in SDK primitives.


In [ ]:
FINAL_REVIEW_TASK = """
Review all three document batches in order.

For each batch:
1. Read the manifest and relevant documents.
2. Preserve cited findings and uncertainty in your working notes and final memo.
3. Preserve any superseded or narrowed assumption instead of silently deleting it.

After Batch 3, write `outputs/compliance_review_memo.md` with:
- executive summary,
- cited findings table,
- open questions,
- recommended next steps for the reviewer.

Reviewer preference for future runs: keep the memo concise, preserve uncertainty
instead of guessing, and separate reusable workflow lessons from document-specific
compliance findings.
""".strip()


def build_reliable_evidence_agent() -> SandboxAgent:
    return SandboxAgent(
        name="Reliable Evidence Review Agent",
        model=MODEL,
        instructions=(
            BASELINE_INSTRUCTIONS
            + "\n\nUse compaction as working context. Use SDK memory for reusable "
            "workflow lessons across runs. Do not treat memory as the system of record "
            "for ACME-specific findings; those belong in the cited memo artifact."
        ),
        default_manifest=build_evidence_manifest(),
        capabilities=[
            Filesystem(),
            Shell(),
            Compaction(),
            workflow_memory(),
        ],
    )


reliable_agent = build_reliable_evidence_agent()
print([type(capability).__name__ for capability in reliable_agent.capabilities])


In [ ]:
EXAMPLE_OUTPUT = """
Compliance Review Memo

Executive summary:
The current evidence supports a control-gap theory, not a final violation finding.
ACME required two approvals for payments above $50,000, and vendor exceptions had
to be logged with Finance Ops. The Northwind Logistics exception appears to have
Legal approval, but Finance Ops approval was pending when payment was released.

Cited findings:
- ACME policy required two approvals for payments above $50,000 and exception
  logging with Finance Ops. Source: ACME-B1-001.
- The Northwind exception was verbally approved, but the note does not show a
  Finance Ops log entry. Source: ACME-B1-002.
- Internal Audit treated missing exception logs as a control gap until resolved.
  Source: ACME-B2-001.
- The approval thread says Finance Ops approval was still pending at release.
  Source: ACME-B2-002.
- The remediation plan requires reconciliation of verbal vendor exceptions from
  Q4 and retrospective control attestations. Source: ACME-B3-001.

Open questions:
- Was Finance Ops approval later completed for Northwind Logistics?
- How many other Q4 verbal exceptions lack retrospective attestations?
- Did the payment release require a compensating control?

Recommended next steps:
1. Reconcile Northwind Logistics against the Finance Ops exception log.
2. Pull all Q4 verbal vendor exceptions into the same review workflow.
3. Ask the reviewer to approve the control-gap classification before escalation.
""".strip()


if RUN_AGENT:
    final_sdk_session = build_compaction_session()
    final_run = await run_in_unix_sandbox(
        reliable_agent,
        FINAL_REVIEW_TASK,
        sdk_session=final_sdk_session,
    )
    print(final_run["final_output"])
    print("\n----- END AGENT OUTPUT -----")
else:
    final_run = {
        "final_output": EXAMPLE_OUTPUT,
        "memo": EXAMPLE_OUTPUT,
        "workspace_files": "Set RUN_AGENT = False to inspect generated sandbox files.",
        "memory_artifacts": {},
        "compaction_checkpoint": None,
    }
    print(EXAMPLE_OUTPUT)
    print("\n----- END EXPECTED OUTPUT SHAPE -----")


## Inspect Generated Artifacts

The final agent response is useful, but the reliability pattern becomes clearer when you inspect the files the sandbox run produced. This section makes the normally hidden state visible:

- the reviewer-facing memo in `outputs/compliance_review_memo.md`,
- generated SDK memory files such as `memories/MEMORY.md` and `memories/memory_summary.md`,
- the workspace files produced by the run, including the session log.

The generated memory artifact is not the compliance memo and should not be treated as investigation truth. It is reusable workflow memory. The `Task Group` heading is the memory system's own grouping label, and the memory generator is steered with `MemoryGenerateConfig.extra_prompt` so it stores workflow lessons rather than ACME-specific findings.

If `RUN_AGENT = False`, this section displays the expected output shape instead of live sandbox artifacts.


In [ ]:
try:
    from IPython.display import Markdown, display
except ImportError:
    Markdown = None
    display = None

memo_text = final_run.get("memo") if "final_run" in globals() else None
workspace_files = final_run.get("workspace_files") if "final_run" in globals() else None
memory_artifacts = final_run.get("memory_artifacts", {}) if "final_run" in globals() else {}
compaction_checkpoint = final_run.get("compaction_checkpoint") if "final_run" in globals() else None

print("Generated workspace files:")
print(workspace_files or "No generated workspace files were captured.")

print("\nMemory and compaction configuration:")
print({
    "final_agent_capabilities": [type(capability).__name__ for capability in reliable_agent.capabilities],
    "sandbox_compaction": "Compaction() is attached to the final agent for automatic context management",
    "sandbox_memory": "Memory(generate=MemoryGenerateConfig(...)) is attached to the final agent",
    "memory_write_policy": "Store reusable workflow lessons, not ACME-specific compliance findings",
    "forced_checkpoint": "OpenAIResponsesCompactionSession.run_compaction({force: True}) after the final review",
    "compaction_model_for_checkpoint": COMPACTION_MODEL,
    "checkpoint_result": compaction_checkpoint,
})

if memory_artifacts:
    for path, text in memory_artifacts.items():
        heading = f"### Generated SDK memory artifact: `{path}`\n\n"
        explanation = (
            "This block is generated by the SDK `Memory()` primitive. It is reusable "
            "workflow memory, not the compliance memo and not the investigation system "
            "of record.\n\n"
            "```text\n----- BEGIN GENERATED SDK MEMORY ARTIFACT -----\n"
        )
        closing = "\n----- END GENERATED SDK MEMORY ARTIFACT -----\n```"
        if display is not None and Markdown is not None:
            display(Markdown(heading + explanation + text + closing))
        else:
            print(f"\nGenerated SDK memory artifact: {path}\n")
            print("----- BEGIN GENERATED SDK MEMORY ARTIFACT -----")
            print(text)
            print("----- END GENERATED SDK MEMORY ARTIFACT -----")
else:
    print("\nNo generated memory artifacts were captured. Set RUN_AGENT = False and rerun the final workflow.")

if memo_text:
    if display is not None and Markdown is not None:
        display(Markdown("## Generated memo: `outputs/compliance_review_memo.md`\n\n" + memo_text))
    else:
        print("\nGenerated memo:\n")
        print(memo_text)
else:
    print("\nNo memo was captured. Set RUN_AGENT = False and rerun the final workflow.")


## Summary: Memory vs. Compaction

The cells above inspect the live run artifacts. This section switches back to a conceptual summary of the two primitives and how they should be used.

| Question | Compaction | Memory |
|---|---|---|
| Main job | Keep a long-running interaction within useful context limits. | Carry reusable lessons into future sandbox-agent runs. |
| Time horizon | Current run or current conversation window. | Future runs and repeated workflows. |
| Typical contents | Condensed working state, recent goals, tool context, unresolved task state. | User preferences, process lessons, recurring pitfalls, project conventions. |
| Human-readable? | Not necessarily. Server-side compaction items may be opaque. | Yes. Sandbox memory artifacts are files the agent can search and read. |
| Source of investigation truth? | No. It is working context. | No. It is workflow memory. |
| What should hold review output? | Generated artifacts such as `outputs/compliance_review_memo.md`. | Generated artifacts such as `outputs/compliance_review_memo.md`. |

The primitives complement each other because they operate at different layers. Compaction helps the active agent continue reviewing many documents without replaying the full transcript. Memory helps the next run remember how to work well in this review process. The memo remains the artifact that humans inspect, validate, and approve.


**Common Pitfall**

Do not treat memory as an unreviewed fact database.

SDK `Memory()` is best used for reusable workflow lessons, preferences, and process corrections. It can remember that the reviewer prefers a concise risk table. It should not be the only place where document-specific compliance findings appear. Keep those findings in generated artifacts with citations.


## Next Steps

Adapt the same pattern to one of these workflows:

- KYC investigation: batches are identity documents, sanctions hits, and analyst notes.
- Incident review: batches are logs, tickets, chat threads, and remediation records.
- Customer escalation: batches are support tickets, call notes, product telemetry, and account history.

The implementation stays the same: use compaction for the long active review, use SDK `Memory()` for workflow lessons, and generate cited artifacts for human review.


## Conclusion

Reliable long-running agents need more than a large prompt. They need clear state boundaries.

In this Cookbook, you built an Evidence Review Agent for a compliance investigation. The baseline agent could inspect documents and write an artifact. Adding `Compaction()` made the long-running review more robust as context grew. Adding SDK `Memory()` gave future runs a way to reuse workflow lessons, with a small generation prompt to keep memory separate from case-specific findings. Running both together produced a stronger harness: compaction for active continuity, memory for cross-run learning, and generated artifacts for human review.

For production systems, keep this separation explicit:

- **Compaction** is working context.
- **Memory** is reusable process knowledge.
- **Artifacts** are what humans inspect, validate, and approve.
